# Feature Engineering for Autoencoder

## Unsupervised Feature Learning

This notebook demonstrates **Autoencoder** architectures for feature extraction and representation learning, covering:

- **Vanilla Autoencoders** - Basic encoder-decoder architecture
- **Variational Autoencoders (VAE)** - Probabilistic feature learning
- **Denoising Autoencoders** - Robust feature extraction
- **Convolutional Autoencoders** - Spatial feature learning
- **Sparse Autoencoders** - Selective feature activation
- **Latent Space Analysis** - Understanding learned representations

### 🔧 Applications:
- **Dimensionality Reduction**: Non-linear feature compression
- **Anomaly Detection**: Reconstruction error-based detection
- **Data Generation**: Sampling from learned latent spaces
- **Feature Learning**: Unsupervised representation discovery
- **Denoising**: Cleaning corrupted data

---

In [ ]:
# Essential Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Notebook: Autoencoder Feature Engineering")
print("=" * 60)

# Generate Diverse Datasets for Autoencoder Training
def create_synthetic_datasets():
    """Create multiple synthetic datasets for autoencoder training"""
    
    datasets = {}
    
    # 1. High-dimensional blob data
    X_blobs, y_blobs = make_blobs(
        n_samples=3000, centers=8, n_features=50, 
        random_state=42, cluster_std=2.0
    )
    datasets['blobs'] = (X_blobs, y_blobs)
    
    # 2. Moon-shaped data (extended to higher dimensions)
    X_moons, y_moons = make_moons(n_samples=2000, noise=0.1, random_state=42)
    # Add random features to make it higher dimensional
    random_features = np.random.randn(2000, 28)
    X_moons_hd = np.hstack([X_moons, random_features])
    datasets['moons'] = (X_moons_hd, y_moons)
    
    # 3. Circular data
    X_circles, y_circles = make_circles(
        n_samples=2000, noise=0.1, factor=0.3, random_state=42
    )
    random_features_circles = np.random.randn(2000, 38)
    X_circles_hd = np.hstack([X_circles, random_features_circles])
    datasets['circles'] = (X_circles_hd, y_circles)
    
    # 4. Spiral data
    n_points = 2000
    noise = 0.1
    t = np.linspace(0, 4*np.pi, n_points)
    x = t * np.cos(t) + noise * np.random.randn(n_points)
    y = t * np.sin(t) + noise * np.random.randn(n_points)
    X_spiral = np.column_stack([x, y])
    random_features_spiral = np.random.randn(n_points, 48)
    X_spiral_hd = np.hstack([X_spiral, random_features_spiral])
    y_spiral = (t > 2*np.pi).astype(int)  # Binary labels
    datasets['spiral'] = (X_spiral_hd, y_spiral)
    
    # 5. Mixed Gaussian data with different variances
    n_samples_per_cluster = 500
    cluster_centers = [(0, 0), (5, 5), (-3, 7), (8, -2), (-6, -4)]
    X_mixed = []
    y_mixed = []
    
    for i, (cx, cy) in enumerate(cluster_centers):
        # Different variances for each cluster
        variance = (i + 1) * 0.5
        cluster_data = np.random.multivariate_normal(
            [cx, cy], [[variance, 0], [0, variance]], n_samples_per_cluster
        )
        X_mixed.append(cluster_data)
        y_mixed.extend([i] * n_samples_per_cluster)
    
    X_mixed = np.vstack(X_mixed)
    random_features_mixed = np.random.randn(len(X_mixed), 28)
    X_mixed_hd = np.hstack([X_mixed, random_features_mixed])
    datasets['mixed_gaussian'] = (X_mixed_hd, np.array(y_mixed))
    
    return datasets

# Create datasets
print("Generating synthetic datasets...")
datasets = create_synthetic_datasets()

# Display dataset information
print("\nDataset Information:")
for name, (X, y) in datasets.items():
    print(f"{name:15} - Shape: {X.shape:15} - Classes: {len(np.unique(y)):2} - Range: [{X.min():.2f}, {X.max():.2f}]")

# Choose primary dataset for training
primary_dataset = 'blobs'
X_raw, y_raw = datasets[primary_dataset]

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"\nUsing '{primary_dataset}' dataset as primary training data")
print(f"Original shape: {X_raw.shape}")
print(f"Scaled data - Mean: {X_scaled.mean():.3f}, Std: {X_scaled.std():.3f}")

# Convert to tensors and create data loader
X_tensor = torch.FloatTensor(X_scaled)
y_tensor = torch.LongTensor(y_raw)

dataset = TensorDataset(X_tensor, y_tensor)
batch_size = 128
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Data loader created with batch size: {batch_size}")
print(f"Number of batches: {len(data_loader)}")

# Visualize data distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, (X, y)) in enumerate(datasets.items()):
    if i >= 6:  # Only show first 6 datasets
        break
        
    ax = axes[i]
    
    # For visualization, use PCA to reduce to 2D
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)
    
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='viridis', alpha=0.6, s=10)
    ax.set_title(f'{name.replace("_", " ").title()}\nShape: {X.shape}')
    ax.set_xlabel('First Principal Component')
    ax.set_ylabel('Second Principal Component')
    plt.colorbar(scatter, ax=ax, shrink=0.8)

plt.suptitle('Synthetic Datasets for Autoencoder Training', fontsize=16)
plt.tight_layout()
plt.show()

print("Data preparation complete!")

In [ ]:
# Autoencoder Architectures Implementation

# 1. Vanilla Autoencoder
class VanillaAutoencoder(nn.Module):
    """Basic autoencoder with fully connected layers"""
    
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[256, 128]):
        super(VanillaAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        encoder_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder (reverse of encoder)
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_reconstructed = self.decode(z)
        return x_reconstructed, z

# 2. Variational Autoencoder (VAE)
class VariationalAutoencoder(nn.Module):
    """Variational Autoencoder with reparameterization trick"""
    
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[256, 128]):
        super(VariationalAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        self.encoder_base = nn.Sequential(*encoder_layers)
        
        # Latent space parameters
        self.mu_layer = nn.Linear(prev_dim, latent_dim)
        self.logvar_layer = nn.Linear(prev_dim, latent_dim)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def encode(self, x):
        h = self.encoder_base(x)
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick"""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_reconstructed = self.decode(z)
        return x_reconstructed, mu, logvar, z

# 3. Denoising Autoencoder
class DenoisingAutoencoder(nn.Module):
    """Denoising autoencoder for robust feature learning"""
    
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[256, 128], noise_factor=0.2):
        super(DenoisingAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.noise_factor = noise_factor
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        
        encoder_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def add_noise(self, x):
        """Add noise to input during training"""
        if self.training:
            noise = torch.randn_like(x) * self.noise_factor
            return x + noise
        return x
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        x_noisy = self.add_noise(x)
        z = self.encode(x_noisy)
        x_reconstructed = self.decode(z)
        return x_reconstructed, z

# 4. Sparse Autoencoder
class SparseAutoencoder(nn.Module):
    """Sparse autoencoder with L1 regularization"""
    
    def __init__(self, input_dim, latent_dim=64, hidden_dims=[256, 128], sparsity_weight=1e-3):
        super(SparseAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.sparsity_weight = sparsity_weight
        
        # Encoder with larger latent dimension for sparsity
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU()
            ])
            prev_dim = hidden_dim
        
        encoder_layers.extend([
            nn.Linear(prev_dim, latent_dim),
            nn.ReLU()  # ReLU for sparsity
        ])
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU()
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_reconstructed = self.decode(z)
        return x_reconstructed, z
    
    def sparsity_loss(self, z):
        """L1 sparsity regularization"""
        return self.sparsity_weight * torch.sum(torch.abs(z))

# Initialize models
input_dim = X_scaled.shape[1]
latent_dim = 16

print("Creating autoencoder models...")

# Create all autoencoder variants
vanilla_ae = VanillaAutoencoder(input_dim, latent_dim=latent_dim).to(device)
vae = VariationalAutoencoder(input_dim, latent_dim=latent_dim).to(device)
denoising_ae = DenoisingAutoencoder(input_dim, latent_dim=latent_dim).to(device)
sparse_ae = SparseAutoencoder(input_dim, latent_dim=latent_dim*2).to(device)

models = {
    'Vanilla AE': vanilla_ae,
    'VAE': vae,
    'Denoising AE': denoising_ae,
    'Sparse AE': sparse_ae
}

print("The autoencoder models created:")
for name, model in models.items():
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  {name}: {param_count:,} parameters")

# Test forward pass
print("\n🔍 Testing forward passes:")
sample_batch = next(iter(data_loader))[0].to(device)[:8]  # First 8 samples

with torch.no_grad():
    for name, model in models.items():
        if name == 'VAE':
            recon, mu, logvar, z = model(sample_batch)
            print(f"  {name}: Input {sample_batch.shape} -> Latent {z.shape} -> Reconstructed {recon.shape}")
        else:
            recon, z = model(sample_batch)
            print(f"  {name}: Input {sample_batch.shape} -> Latent {z.shape} -> Reconstructed {recon.shape}")

print("\nThe models tested successfully!")

# Feature Engineering with PyTorch for Autoencoder

## Unsupervised Feature Learning through Reconstruction

This notebook demonstrates **Autoencoder** architectures for feature engineering.

- **Vanilla Autoencoders** for basic dimensionality reduction
- **Variational Autoencoders (VAE)** for probabilistic features
- **Denoising Autoencoders** for robust feature learning
- **Convolutional Autoencoders** for image features
- **Latent space exploration** and interpolation
- **Feature visualization** and analysis

### 🔧 Autoencoder Types:
- **Simple Autoencoder**: Basic encoder-decoder structure
- **Convolutional Autoencoder**: For image data
- **Variational Autoencoder**: Probabilistic latent space
- **Denoising Autoencoder**: Robust to noise and corruption
- **Sparse Autoencoder**: Encourages sparse representations

---